# Phase 1: Model Fine-Tuning & Quantization

Run this notebook in Google Colab (Free T4 or Pro) to fine-tune Llama 3.1 8B using Unsloth.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets huggingface_hub

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 
dtype = None # None for auto detection
load_in_4bit = True # 4bit quantization to save VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset

# Sample dataset processing (mapping to Llama 3 format)
# You might need to adjust the dataset and mapping based on the exact structure of medical_meadow_wikidoc_patient_information
dataset = load_dataset("medalpaca/medical_meadow_wikidoc_patient_information", split="train")

prompt_template = """<|start_header_id|>system<|end_header_id|>

You are a clinical-grade medical AI assistant. Answer using only the provided context. If the context is insufficient, state that you do not know.<|eot_id|><|start_header_id|>user<|end_header_id|>

{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{output}<|eot_id|>"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for i, o in zip(inputs, outputs):
        # Must add EOS_TOKEN
        text = prompt_template.format(input=i, output=o) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase for a full run (e.g., 1-2 epochs)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# Export to GGUF
# This will export to Q4_K_M for Mac inference
model.save_pretrained_gguf("fine_tuned_medical_llama3_q4", tokenizer, quantization_method = "q4_k_m")

print("Finished exporting. Download 'fine_tuned_medical_llama3_q4.gguf' to your Mac.")